## Phase 1: Data Pipeline

Goal of this phase: load the Warehouse Object Detection dataset that I uploaded to Drive, understand what we're working with, and build a PyTorch Dataset that yields `(cropped_image, class_label)`
pairs ready for classification.

We're framing the task as **crop-based single-label classification**: each bounding box in the dataset becomes one training sample, cropped from its parent image and labeled with its class.

In [1]:
# mount drive and import basics
from google.colab import drive
drive.mount('/content/drive')

import os
import json
import random
from pathlib import Path
from collections import Counter

import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

Mounted at /content/drive


In [2]:
# all paths in one place
DRIVE_ROOT = Path('/content/drive/MyDrive/Colab Notebooks/AdvancedDL') # Manually Uploaded
DATA_ROOT = DRIVE_ROOT / 'coco'

IMG_DIRS = {
    'train': DATA_ROOT / 'images' / 'train',
    'val':   DATA_ROOT / 'images' / 'val',
    'test':  DATA_ROOT / 'images' / 'test',
}
ANN_FILES = {
    'train': DATA_ROOT / 'annotations' / 'instances_train.json',
    'val':   DATA_ROOT / 'annotations' / 'instances_val.json',
    'test':  DATA_ROOT / 'annotations' / 'instances_test.json',
}

# sanity check
for split, p in IMG_DIRS.items():
    print(f'{split} images: {p.exists()} - {p}')
for split, p in ANN_FILES.items():
    print(f'{split} anns:   {p.exists()} - {p}')

train images: True - /content/drive/MyDrive/Colab Notebooks/AdvancedDL/coco/images/train
val images: True - /content/drive/MyDrive/Colab Notebooks/AdvancedDL/coco/images/val
test images: True - /content/drive/MyDrive/Colab Notebooks/AdvancedDL/coco/images/test
train anns:   True - /content/drive/MyDrive/Colab Notebooks/AdvancedDL/coco/annotations/instances_train.json
val anns:   True - /content/drive/MyDrive/Colab Notebooks/AdvancedDL/coco/annotations/instances_val.json
test anns:   True - /content/drive/MyDrive/Colab Notebooks/AdvancedDL/coco/annotations/instances_test.json


## Approach A — Manual JSON parsing

Let's parse the annotations by hand to understand
the data structure. COCO JSON has 4 top-level keys: `images`, `annotations`,
`categories`, `info`.

This was understood after a back and ford using the read.me from the dataset. A suggestion was made of using a library pycocotools but I am first navigating a manual solution

In [3]:
# load the train annotations manually with plain json
with open(ANN_FILES['train']) as f:
    coco_raw = json.load(f)

print('Top-level keys:', list(coco_raw.keys()))
print(f"Images:      {len(coco_raw['images'])}")
print(f"Annotations: {len(coco_raw['annotations'])}")
print(f"Categories:  {len(coco_raw['categories'])}")

# peek at one of each
print('\nExample image entry:')
print(coco_raw['images'][0])
print('\nExample annotation entry:')
print(coco_raw['annotations'][0])
print('\nExample category entry:')
print(coco_raw['categories'][0])

Top-level keys: ['info', 'licenses', 'categories', 'images', 'annotations']
Images:      4363
Annotations: 53984
Categories:  25

Example image entry:
{'id': 1, 'file_name': 'warehouse_002370.png', 'width': 512, 'height': 512, 'date_captured': '', 'license': 1}

Example annotation entry:
{'id': 1, 'image_id': 1, 'category_id': 15, 'bbox': [0, 401.0, 217.0, 110.0], 'area': 23870.0, 'iscrowd': 0, 'segmentation': []}

Example category entry:
{'id': 1, 'name': 'box', 'supercategory': 'container'}
